In [ ]:
!pip install spacy nltk requests


In [ ]:
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 31.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving pubmed_ALL_cancer_environment_20251214_010208.csv to pubmed_ALL_cancer_environment_20251214_010208.csv


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv('pubmed_ALL_cancer_environment_20251214_010208.csv')

In [ ]:
import pandas as pd
import requests
import spacy
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt")
nlp = spacy.load("en_core_web_sm")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
!pip install tqdm


In [ ]:
import pandas as pd
import requests
import spacy
import re
from tqdm import tqdm

nlp = spacy.load("en_core_web_sm")


In [ ]:
FINAL_DISEASE_NAMES = {
    "heart_disease": [
        "heart","heart disease","cardiovascular disease","cardiovascular diseases","cvd","coronary heart disease","coronary artery disease","ischemic heart disease", "atherosclerotic cardiovascular disease","ascvd","myocardial infarction","heart attack","acute coronary syndrome","angina pectoris","heart failure","congestive heart failure","hypertensive heart disease","peripheral artery disease","peripheral vascular disease","stroke","cerebrovascular disease","cardiac disease","vascular disease"
    ],

    "diabetes": [
        "diabetes", "diabetes mellitus", "type 1 diabetes","type i diabetes","t1d","type 2 diabetes","type ii diabetes","t2d","gestational diabetes","adult-onset diabetes", "juvenile diabetes","hyperglycemia", "chronic hyperglycemia","insulin resistance","impaired glucose tolerance","impaired fasting glucose","prediabetes","metabolic disease","metabolic disorder","glucose metabolism disorder", "diabetic disease"
    ],

    "lung_disease": [
        "lung disease",
        "chronic lung disease",
        "respiratory disease",
        "respiratory diseases",
        "chronic respiratory disease",
        "pulmonary disease",
        "pulmonary disorders",
        "chronic obstructive pulmonary disease",
        "copd",
        "asthma",
        "chronic bronchitis",
        "emphysema",
        "interstitial lung disease",
        "ild",
        "pulmonary fibrosis",
        "restrictive lung disease",
        "obstructive lung disease",
        "airway disease",
        "lower respiratory disease",
        "respiratory illness"
    ],

    "cancer": [
        "cancer",
        "cancers",
        "malignancy",
        "malignancies",
        "malignant neoplasm",
        "neoplasm",
        "neoplasms",
        "tumor",
        "tumors",
        "carcinoma",
        "adenocarcinoma",
        "squamous cell carcinoma",
        "sarcoma",
        "lymphoma",
        "leukemia",
        "solid tumor",
        "metastatic cancer",
        "advanced cancer",
        "oncologic disease",
        "oncological disease",

    "Adenocarcinoma", "Basal Cell Carcinoma", "Squamous Cell Carcinoma",
    "Renal Cell Carcinoma", "Transitional Cell Carcinoma", "Ductal Carcinoma In Situ",
    "Invasive Ductal Carcinoma", "Invasive Lobular Carcinoma", "Hepatocellular Carcinoma",
    "Adrenocortical Carcinoma", "Bronchioloalveolar Carcinoma", "Merkel Cell Carcinoma",
    "Sebaceous Carcinoma", "Acinar Cell Carcinoma", "Papillary Carcinoma",
    "Follicular Carcinoma", "Medullary Carcinoma", "Mucoepidermoid Carcinoma",

    "Osteosarcoma", "Chondrosarcoma", "Ewing Sarcoma", "Liposarcoma",
    "Rhabdomyosarcoma", "Leiomyosarcoma", "Angiosarcoma", "Kaposi Sarcoma",
    "Synovial Sarcoma", "Fibrosarcoma", "Malignant Fibrous Histiocytoma",
    "Alveolar Soft Part Sarcoma", "Clear Cell Sarcoma", "Desmoplastic Small Round Cell Tumor",
    "Epithelioid Sarcoma", "Gastrointestinal Stromal Tumor",

    "Acute Lymphoblastic Leukemia", "Acute Myeloid Leukemia",
    "Chronic Lymphocytic Leukemia", "Chronic Myeloid Leukemia",
    "Hairy Cell Leukemia", "Acute Promyelocytic Leukemia",
    "Chronic Myelomonocytic Leukemia", "Juvenile Myelomonocytic Leukemia",
    "Large Granular Lymphocytic Leukemia", "Blastic Plasmacytoid Dendritic Cell Neoplasm",

    # Lymphomas
    "Hodgkin Lymphoma", "Non-Hodgkin Lymphoma", "Diffuse Large B-Cell Lymphoma",
    "Follicular Lymphoma", "Mantle Cell Lymphoma", "Burkitt Lymphoma",
    "Marginal Zone Lymphoma", "Primary Central Nervous System Lymphoma",
    "Primary Mediastinal B-Cell Lymphoma", "Anaplastic Large Cell Lymphoma",
    "Peripheral T-Cell Lymphoma", "Cutaneous T-Cell Lymphoma",
    "Mycosis Fungoides", "Sézary Syndrome", "Lymphoplasmacytic Lymphoma",
    "Primary Effusion Lymphoma", "Post-Transplant Lymphoproliferative Disorder",

    "Glioblastoma", "Astrocytoma", "Oligodendroglioma", "Ependymoma",
    "Medulloblastoma", "Meningioma", "Pituitary Adenoma", "Craniopharyngioma",
    "Schwannoma", "Pineoblastoma", "Pineocytoma", "Chordoma",
    "Central Neurocytoma", "Dysembryoplastic Neuroepithelial Tumor",

    "Neuroblastoma", "Retinoblastoma", "Wilms Tumor", "Rhabdoid Tumor",
    "Hepatoblastoma", "Germ Cell Tumor", "Osteosarcoma (Pediatric)",
    "Ewing Sarcoma (Pediatric)", "Rhabdomyosarcoma (Pediatric)",

    "Ovarian Cancer", "Cervical Cancer", "Endometrial Cancer",
    "Uterine Cancer", "Vulvar Cancer", "Vaginal Cancer",
    "Gestational Trophoblastic Disease", "Choriocarcinoma",
    "Ovarian Germ Cell Tumor", "Fallopian Tube Cancer",


    "Prostate Cancer", "Testicular Cancer", "Bladder Cancer",
    "Kidney Cancer", "Penile Cancer", "Urethral Cancer",
    "Renal Pelvis Cancer", "Ureter Cancer",

    "Colorectal Cancer", "Colon Cancer", "Rectal Cancer",
    "Stomach Cancer", "Esophageal Cancer", "Pancreatic Cancer",
    "Liver Cancer", "Gallbladder Cancer", "Bile Duct Cancer",
    "Anal Cancer", "Small Intestine Cancer", "Appendiceal Cancer",
    "Peritoneal Cancer", "Gastroesophageal Junction Cancer",

    "Oral Cancer", "Tongue Cancer", "Throat Cancer",
    "Laryngeal Cancer", "Nasopharyngeal Cancer",
    "Oropharyngeal Cancer", "Hypopharyngeal Cancer",
    "Salivary Gland Cancer", "Paranasal Sinus Cancer",
    "Nasal Cavity Cancer", "Middle Ear Cancer",


    "Lung Cancer", "Small Cell Lung Cancer", "Non-Small Cell Lung Cancer",
    "Mesothelioma", "Thymoma", "Thymic Carcinoma",


    "Thyroid Cancer", "Adrenal Cancer", "Parathyroid Cancer",
    "Pituitary Cancer", "Pancreatic Neuroendocrine Tumor",
    "Carcinoid Tumor", "Pheochromocytoma",

    "Melanoma", "Skin Cancer (Non-Melanoma)",
    "Dermatofibrosarcoma Protuberans", "Cutaneous Lymphoma",

    "Bone Cancer", "Soft Tissue Sarcoma", "Giant Cell Tumor of Bone",

    "Multiple Myeloma", "Waldenström Macroglobulinemia",
    "Myelodysplastic Syndromes", "Myeloproliferative Neoplasms",
    "Polycythemia Vera", "Essential Thrombocythemia",
    "Primary Myelofibrosis", "Systemic Mastocytosis",

    "Desmoid Tumor", "Pseudomyxoma Peritonei",
    "Ameloblastoma", "Adamantinoma", "Extragonadal Germ Cell Tumor",
    "Phyllodes Tumor", "Male Breast Cancer", "Inflammatory Breast Cancer",
    "Triple-Negative Breast Cancer", "HER2-Positive Breast Cancer",
    "Metastatic Cancer", "Carcinoma of Unknown Primary"

    ]
}



In [ ]:


ENVIRONMENTAL_FACTORS = {
    "air_pollution": [
        "pm2.5", "pm10", "ultrafine particles", "ufp", "particulate matter",
        "nitrogen dioxide", "no2", "sulfur dioxide", "so2", "ozone", "o3",
        "carbon monoxide", "co", "volatile organic compounds", "vocs",
        "benzene", "formaldehyde", "toluene", "xylene", "ethylbenzene",
        "polycyclic aromatic hydrocarbons", "pahs", "dioxins", "furans",
        "black carbon", "elemental carbon", "organic carbon",
        "aerosol", "secondary inorganic aerosols",
        "air pollution", "diesel exhaust", "vehicle emissions",
        "industrial emissions", "power plant emissions", "dust",
        "road dust", "particles", "brake wear particles",
        "biomass burning", "wildfire smoke", "agricultural burning",
        "cooking fumes", "cigarette smoke", "secondhand smoke",
        "wood burning", "coal burning", "kerosene heaters",
        "indoor air pollution", "outdoor air pollution", "ambient air quality",
        "air quality index", "aqi", "air pollution exposure",
        "urban air pollution", "rural air pollution", "industrial air pollution",
        "transportation air pollution", "residential air pollution",
        "occupational air pollution", "tobacco smoke",
        "environmental smoke", "airborne particles", "airborne contaminants",
        "airborne toxins", "airborne chemicals", "airborne pollutants",
        "air pollution mixture", "air pollution cocktail", "air pollution soup"
    ],

    "chemical_exposures": [
        "heavy metals", "lead", "pb", "mercury", "hg", "arsenic", "as",
        "cadmium", "cd", "chromium", "cr", "nickel", "ni", "manganese", "mn",
        "aluminum", "al", "copper", "cu", "zinc", "zn", "selenium", "se",
        "iron", "fe", "cobalt", "co", "vanadium", "v", "antimony", "sb",
        "thallium", "tl", "beryllium", "be", "uranium", "u", "plutonium", "pu",
        "persistent organic pollutants", "pops", "polychlorinated biphenyls", "pcbs",
        "organochlorine pesticides", "ddt", "dieldrin", "chlordane", "heptachlor",
        "aldrin", "endrin", "toxaphene", "mirex", "hexachlorobenzene", "hcb",
        "perfluorinated compounds", "pfoa", "pfos", "pfas", "forever chemicals",
        "brominated flame retardants", "pbdes", "tbbpa", "hbcds",
        "phthalates", "dehp", "dbp", "bbp", "dibp", "dimp", "didp", "dinp",
        "bisphenols", "bpa", "bps", "bpf", "bpaf", "bpb", "bpe", "bpp",
        "parabens", "methylparaben", "ethylparaben", "propylparaben", "butylparaben",
        "triclosan", "triclocarban", "benzophenones", "oxybenzone", "avobenzone",
        "synthetic musks", "galaxolide", "tonalide", "cashmeran", "celestolide",
        "alkylphenols", "nonylphenol", "octylphenol", "bisphenol a", "bisphenol s",
        "solvents", "trichloroethylene", "tetrachloroethylene", "perchloroethylene",
        "methylene chloride", "chloroform", "carbon tetrachloride", "benzene",
        "toluene", "xylene", "styrene", "acetone", "ethanol", "methanol",
        "glycol ethers", "ethylene glycol", "propylene glycol",
        "plasticizers", "adipates", "citrates", "trimellitates", "sebacates",
        "antimicrobials", "quaternary ammonium compounds", "quats",
        "nanomaterials", "carbon nanotubes", "titanium dioxide nanoparticles",
        "silver nanoparticles", "quantum dots", "fullerenes", "graphene",
        "microplastics", "nanoplastics", "plastic particles", "synthetic fibers",
        "industrial chemicals", "phthalate esters", "alkylphenol ethoxylates",
        "polychlorinated naphthalenes", "chlorinated paraffins", "siloxanes",
        "acrylamide", "acrylonitrile", "vinyl chloride", "ethylene oxide",
        "propylene oxide", "butadiene", "isoprene", "formaldehyde", "acetaldehyde",
        "acrylic acid", "methacrylic acid", "maleic anhydride", "phthalic anhydride"
    ],

    "contaminants": [
        "drinking water contamination", "water pollution", "water quality",
        "groundwater contamination", "surface water contamination",
        "water disinfection byproducts", "trihalomethanes", "thms",
        "haloacetic acids", "haas", "chloroform", "bromodichloromethane",
        "dibromochloromethane", "bromoform", "chloramines", "chlorine dioxide",
        "ozonation byproducts", "bromate", "chlorite", "chlorate",
        "arsenic in water", "lead in water", "mercury in water",
        "cadmium in water", "chromium in water", "nitrate contamination",
        "nitrite contamination", "fluoride levels", "uranium in water",
        "radon in water", "perchlorate contamination", "pesticides in water",
        "herbicides in water", "insecticides in water", "fungicides in water",
        "pharmaceutical residues", "antibiotics in water", "hormones in water",
        "endocrine disruptors in water", "microplastics in water",
        "algae toxins", "cyanotoxins", "microcystins", "cylindrospermopsin",
        "anatoxin-a", "saxitoxins", "bacterial contamination",
        "viral contamination", "parasitic contamination",
        "legionella bacteria", "e. coli contamination", "coliform bacteria",
        "industrial discharge", "agricultural runoff", "urban runoff",
        "sewage contamination", "wastewater effluent", "stormwater runoff",
        "fracking fluids", "hydraulic fracturing chemicals",
        "coal ash contamination", "mine drainage", "acid mine drainage",
        "saltwater intrusion", "desalination brine", "water scarcity",
        "water stress", "water access", "water infrastructure",
        "pipe corrosion", "lead pipes", "copper pipes", "pvc pipes",
        "water treatment chemicals", "coagulants", "flocculants",
        "disinfectants", "corrosion inhibitors", "scale inhibitors",
        "water hardness", "water softeners", "water filtration",
        "reverse osmosis", "distillation", "activated carbon filtration", "pesticide residues", "herbicide residues", "insecticide residues",
        "fungicide residues", "antibiotic residues", "growth hormone residues",
        "veterinary drug residues", "heavy metals in food", "lead in food",
        "mercury in fish", "arsenic in rice", "cadmium in shellfish",
        "aflatoxins", "ochratoxin", "fumonisins", "zearalenone",
        "deoxynivalenol", "patulin", "mycotoxins", "marine biotoxins",
        "ciguatoxin", "saxitoxin", "domoic acid", "histamine",
        "food additives", "preservatives", "artificial colors",
        "artificial flavors", "artificial sweeteners", "msg",
        "sodium nitrite", "sodium nitrate", "sulfites", "sulfur dioxide",
        "bha", "bht", "tbho", "propyl gallate", "benzoates", "sorbates",
        "acrylamide", "furan", "maillard reaction products",
        "heterocyclic amines", "polycyclic aromatic hydrocarbons in food",
        "advanced glycation end products", "trans fats", "hydrogenated oils",
        "processed meats", "red meat consumption", "charred meat",
        "grilled meat", "fried food", "ultra-processed foods",
        "fast food", "junk food", "sugar-sweetened beverages",
        "high-fructose corn syrup", "artificial sweetener consumption",
        "saccharin", "aspartame", "sucralose", "acesulfame potassium",
        "neotame", "advantame", "food packaging migrants",
        "bpa from cans", "phthalates from packaging", "pfas from packaging",
        "styrene from containers", "vinyl chloride from packaging",
        "irradiated foods", "genetically modified foods", "gmo foods",
        "organic foods", "conventional foods", "local foods",
        "imported foods", "food deserts", "food swamps", "food environment",
        "food access", "food insecurity", "nutritional deficiencies",
        "vitamin deficiencies", "mineral deficiencies", "calorie intake",
        "dietary patterns", "mediterranean diet", "dash diet",
        "western diet", "prudent diet", "plant-based diet",
        "vegetarian diet", "vegan diet", "ketogenic diet",
        "low-carb diet", "high-protein diet", "intermittent fasting",
        "calorie restriction", "overeating", "binge eating",
        "emotional eating", "stress eating", "eating disorders",
        "food allergies", "food intolerances", "celiac disease",
        "gluten sensitivity", "lactose intolerance", "fructose malabsorption", "contaminated soil", "soil pollution", "land pollution",
        "brownfields", "superfund sites", "hazardous waste sites",
        "industrial sites", "former industrial sites", "landfill sites",
        "waste disposal", "incineration sites", "mining sites",
        "smelter sites", "manufacturing sites", "chemical plants",
        "oil refineries", "gas stations", "dry cleaners",
        "heavy metals", "lead", "arsenic",
        "cadmium", "chromium", "mercury",
        "pesticides", "herbicides", "pcbs",
        "dioxins", "pahs", "petroleum hydrocarbons",
        "btex compounds", "chlorinated", "radionuclides",
        "uranium", "radium", "thorium",
        "asbestos", "silica", "asbestos fibers",
        "soil erosion", "soil degradation", "desertification",
        "soil salinization", "soil acidification", "soil compaction",
        "topsoil loss", "fertile soil loss", "agricultural soil quality",
        "urban soil quality", "residential soil quality",
        "bioremediation", "phytoremediation", "soil vapor extraction",
        "soil amendments", "compost", "manure", "biosolids",
        "sewage sludge", "industrial sludge", "wastewater sludge",
        "soil dust", "wind-blown dust", "dust storms", "saharan dust",
        "asian dust", "desert dust", "road dust", "construction dust",
        "soil ingestion", "geophagia", "pica", "soil contact",
        "dermal exposure", "inhalation exposure", "ingestion exposure"
    ],

    "climate_factors": [
        "climate change", "global warming", "temperature increase",
        "heat waves", "extreme heat", "heat stress", "heat exposure",
        "cold spells", "extreme cold", "cold stress", "cold exposure",
        "temperature variability", "temperature extremes",
        "urban heat island", "heat island effect", "heat vulnerability",
        "heat-related illness", "heat stroke", "heat exhaustion",
        "cold-related illness", "hypothermia", "frostbite",
        "humidity levels", "relative humidity", "absolute humidity",
        "dew point", "heat index", "wet bulb temperature",
        "humidex", "apparent temperature", "wind chill",
        "wind speed", "wind patterns", "prevailing winds",
        "atmospheric pressure", "barometric pressure",
        "air pressure changes", "weather fronts", "storm systems",
        "extreme weather events", "hurricanes", "typhoons", "cyclones",
        "tornadoes", "floods", "flash floods", "storm surges",
        "droughts", "water scarcity", "wildfires", "forest fires",
        "bushfires", "dust storms", "sandstorms", "hailstorms",
        "blizzards", "snowstorms", "ice storms", "freezing rain",
        "seasonal changes", "seasonal variations", "seasonal patterns",
        "pollen seasons", "allergy seasons", "mold seasons",
        "seasonal affective disorder", "sad", "winter blues",
        "summer heat", "monsoon season", "rainy season", "dry season",
        "el nino", "la nina", "enso", "climate oscillations",
        "sea level rise", "coastal flooding", "saltwater intrusion",
        "climate refugees", "environmental migration", "climate migration",
        "climate adaptation", "climate resilience", "climate vulnerability",
        "climate justice", "environmental justice", "climate equity"
    ],

    "built_environment": [
        "urban design", "city planning", "urban planning",
        "land use", "zoning", "mixed-use development",
        "residential density", "population density", "housing density",
        "building density", "urban sprawl", "suburban sprawl",
        "compact cities", "smart growth", "transit-oriented development",
        "walkability", "walkable neighborhoods", "pedestrian-friendly",
        "sidewalk availability", "crosswalk safety", "street connectivity",
        "grid street pattern", "cul-de-sacs", "dead-end streets",
        "bikeability", "bicycle infrastructure", "bike lanes",
        "bike paths", "bike sharing", "cycling facilities",
        "public transit", "bus service", "train service",
        "subway service", "light rail", "commuter rail",
        "transit access", "transit frequency", "transit reliability",
        "parking availability", "parking costs", "car dependency",
        "traffic congestion", "traffic volume", "traffic speed",
        "road network", "highway proximity", "major road proximity",
        "traffic noise", "road traffic noise", "aircraft noise",
        "railway noise", "industrial noise", "construction noise",
        "noise pollution", "noise exposure", "noise levels",
        "light pollution", "artificial light at night", "street lighting",
        "security lighting", "advertising lighting", "skyglow",
        "light trespass", "glare", "circadian disruption",
        "green spaces", "parks", "playgrounds", "recreational areas",
        "open spaces", "public spaces", "community gardens",
        "urban forestry", "street trees", "tree canopy cover",
        "vegetation cover", "green infrastructure", "green roofs",
        "green walls", "rain gardens", "bioswales",
        "blue spaces", "water features", "lakes", "rivers", "ponds",
        "fountains", "beaches", "waterfront access",
        "food environment", "food deserts", "food swamps",
        "grocery store access", "supermarket proximity",
        "fast food outlets", "convenience stores", "restaurant density",
        "alcohol outlets", "liquor", "bars", "pubs",
        "tobacco outlets", "vape", "smoking",
        "healthcare access", "hospital proximity", "clinic proximity",
        "pharmacy access", "primary care access", "specialist access",
        "school environment", "school quality", "school proximity",
        "playground safety", "play equipment", "sports facilities",
        "community centers", "libraries", "cultural facilities",
        "crime rates", "perceived safety", "neighborhood safety",
        "police presence", "security measures", "surveillance",
        "social cohesion", "neighborhood trust", "social capital",
        "community engagement", "civic participation", "volunteering",
        "housing quality", "housing conditions", "overcrowding",
        "housing affordability", "rent burden", "mortgage stress",
        "homelessness", "temporary housing", "shelter access",
        "indoor environment", "indoor air quality", "ventilation",
        "mold exposure", "dampness", "water damage", "leaks",
        "pest infestation", "cockroaches", "rodents", "bed bugs",
        "asbestos", "lead", "radon",
        "formaldehyde", "voc", "indoor allergens",
        "dust", "pet dander", "pollen",
        "heating systems", "cooling systems", "air conditioning",
        "humidity control", "temperature control", "thermal comfort"
    ],

    "occupational_exposures": [
        "workplace exposures", "occupational hazards", "job-related exposures",
        "industrial chemicals", "industrial solvents", "industrial dusts",
        "industrial fumes", "industrial gases", "industrial vapors",
        "manufacturing exposures", "factory work", "assembly line work",
        "chemical industry", "petrochemical industry", "pharmaceutical industry",
        "agricultural work", "farming", "pesticide application",
        "herbicide application", "fertilizer handling", "animal handling",
        "grain dust", "silo gases", "farm machinery", "tractor operation",
        "construction work", "building trades", "carpentry", "masonry",
        "electrical work", "plumbing", "painting", "roofing",
        "asbestos", "lead abatement", "mold remediation",
        "welding fumes", "soldering fumes", "metalworking fluids",
        "mining exposures", "coal mining", "hard rock mining",
        "silica dust", "coal dust", "metal dust", "mineral dust",
        "quarry work", "sand and gravel operations", "stone cutting",
        "textile industry", "cotton dust", "wool dust", "synthetic fibers",
        "dye exposure", "fabric finishing", "textile printing",
        "healthcare exposures", "hospital work", "nursing", "doctors",
        "dental work", "veterinary work", "laboratory work",
        "antimicrobial agents", "disinfectants", "sterilants",
        "anesthetic gases", "chemotherapy drugs", "radiation exposure",
        "latex allergy", "needlestick injuries", "biological hazards",
        "transportation work", "truck driving", "bus driving",
        "taxi driving", "delivery work", "warehouse work",
        "diesel exhaust", "engine emissions", "traffic exposure",
        "office work", "sedentary work", "desk job", "computer work",
        "call center work", "customer service", "administrative work",
        "ergonomic hazards", "repetitive strain", "musculoskeletal disorders",
        "psychosocial factors", "work stress", "job strain",
        "workload demands", "job control", "decision latitude",
        "social support at work", "workplace relationships",
        "organizational justice", "fair treatment", "discrimination",
        "harassment", "bullying", "violence at work",
        "work hours", "shift work", "night work", "rotating shifts",
        "long hours", "overtime", "work-life balance",
        "job insecurity", "temporary work", "contract work",
        "precarious employment", "unemployment", "job loss",
        "retirement", "pension status", "economic security"
    ],

    "radiation_exposures": [
        "ionizing radiation", "non-ionizing radiation", "radiation exposure",
        "natural radiation", "cosmic radiation", "terrestrial radiation",
        "radon gas", "radon exposure", "radon concentrations",
        "indoor radon", "basement radon", "soil radon",
        "radon mitigation", "radon testing", "radon remediation",
        "medical radiation", "diagnostic radiation", "x-rays",
        "ct scans", "computed tomography", "mammography",
        "fluoroscopy", "interventional radiology", "nuclear medicine",
        "radiation therapy", "radiotherapy", "brachytherapy",
        "occupational radiation", "radiation workers", "nuclear workers",
        "medical radiation workers", "dental radiation", "industrial radiography",
        "airline crew radiation", "cosmic radiation at altitude",
        "frequent flyer radiation", "space radiation", "solar radiation",
        "solar particle events", "galactic cosmic rays",
        "ultraviolet radiation", "uv radiation", "uv-a", "uv-b", "uv-c",
        "sun exposure", "solar ultraviolet", "tanning beds", "sunlamps",
        "sunburn", "sun protection", "sunscreen use", "sunscreen chemicals",
        "visible light", "blue light", "led light", "fluorescent light",
        "infrared radiation", "heat radiation", "thermal radiation",
        "microwave radiation", "microwave ovens", "microwave devices",
        "radiofrequency radiation", "rf radiation", "wireless radiation",
        "cell phone radiation", "mobile phone radiation", "smartphone radiation",
        "wifi radiation", "bluetooth radiation", "cellular towers",
        "base stations", "antenna arrays", "broadcast towers",
        "radio towers", "tv towers", "radar installations",
        "extremely low frequency fields", "elf fields", "power lines",
        "electrical wiring", "electrical appliances", "household appliances",
        "electric blankets", "waterbed heaters", "electric heating pads",
        "geomagnetic fields", "earth's magnetic field", "magnetic storms",
        "solar flares", "geomagnetic activity", "space weather"
    ],




"other_FACTORS":[
    "traffic pollution", "roadway proximity", "diesel exhaust particles",
    "fine particulate matter", "pm2.5 exposure", "ozone exposure",
    "nitrogen oxides", "sulfur dioxide", "carbon monoxide",
    "indoor air pollution", "secondhand smoke", "environmental tobacco smoke",

    "arsenic exposure", "lead exposure", "cadmium exposure",
    "organophosphate pesticides", "phthalate exposure", "bpa exposure",
    "pfoa exposure", "pfc exposure", "persistent organic pollutants",

    "temperature extremes", "heat waves", "cold spells",
    "temperature variability", "barometric pressure changes",

    "chronic stress", "work stress", "financial stress",
    "social isolation", "depression", "anxiety",

    "traffic noise", "aircraft noise", "railway noise",
    "occupational noise", "noise annoyance",

    "walkability", "green space access", "recreational facilities",
    "food environment", "fast food density", "alcohol outlet density",

    "shift work", "night work", "sedentary work",
    "job strain", "workplace bullying",

    "hard water", "soft water", "mineral content",
    "fluoride levels", "chlorination byproducts",

    "ionizing radiation", "medical radiation", "ultraviolet radiation",

    "chronic inflammation", "oxidative stress", "immune activation",
    "infectious burden", "dental infections", "periodontal disease",

    "sodium intake", "potassium intake", "trans fats",
    "saturated fats", "processed meats", "sugar-sweetened beverages"

    "phthalates", "bisphenol a", "organotins", "organochlorine pesticides",
    "pcbs", "dioxins", "perfluorinated compounds", "polybrominated diphenyl ethers",
    "triclosan", "parabens", "benzophenones",

    "pm2.5", "ozone", "nitrogen dioxide", "traffic-related pollution",

    "food deserts", "fast food availability", "supermarket access",
    "walkability", "recreational facilities", "neighborhood safety",
    "sedentary work", "shift work", "night work", "work stress",
    "temperature extremes", "urban heat island",

    "arsenic in water", "nitrate in water", "fluoride in water",
    "artificial light at night", "circadian disruption",
    "gut microbiome", "antibiotic use", "infections",
    "inflammation", "oxidative stress",
    "depression", "stress", "sleep deprivation", "social isolation",
    "sugar-sweetened beverages", "processed foods", "artificial sweeteners",
    "food additives", "pesticide residues", "persistent organic pollutants in food",
    "pm2.5", "pm10", "ozone", "nitrogen dioxide", "sulfur dioxide",
    "traffic pollution", "diesel exhaust", "indoor air pollution",
    "secondhand smoke", "biomass smoke", "wildfire smoke",
    "mold spores", "dust mites", "cockroach allergens", "pet dander",
    "pollen", "endotoxins", "volatile organic compounds",
    "asbestos", "silica dust", "coal dust", "grain dust", "cotton dust",
    "isocyanates", "welding fumes", "wood dust", "metal fumes",
    "chemical vapors", "industrial gases",
    "housing conditions", "dampness", "mold growth", "ventilation",
    "heating systems", "cooling systems", "air conditioning",
    "humidity control", "indoor allergens",
    "temperature extremes", "humidity", "thunderstorms", "pollen seasons",
    "wildfire seasons", "dust storms",

    "legionella", "hot tub aerosols", "humidifier use",
    "water damage", "flooding",
    "radon gas", "indoor radon", "basement radon",
    "respiratory infections", "viral infections", "bacterial infections",
    "fungal infections", "tuberculosis", "influenza",
    "chronic inflammation", "immune response",
    "stress", "anxiety", "depression",
    "antioxidant intake", "vitamin d", "omega-3 fatty acids",
    "processed meats", "fried foods",
    "benzene", "formaldehyde", "asbestos", "arsenic", "chromium vi",
    "nickel compounds", "cadmium", "beryllium", "vinyl chloride",
    "ethylene oxide", "acrylamide", "acrylonitrile",
    "polycyclic aromatic hydrocarbons", "heterocyclic amines",
    "nitrosamines", "aflatoxins", "pcbs", "dioxins",
    "organochlorine pesticides", "chlorinated solvents",
    "pm2.5", "diesel exhaust", "outdoor air pollution",
    "indoor air pollution", "secondhand smoke", "radon gas",
    "ionizing radiation", "ultraviolet radiation", "medical radiation",
    "radon exposure", "gamma radiation", "x-rays",
    "radiofrequency radiation", "extremely low frequency fields",
    "asbestos exposure", "silica dust", "wood dust", "leather dust",
    "rubber industry chemicals", "painting chemicals",
    "metalworking fluids", "shift work", "night work",
    "hepatitis b virus", "hepatitis c virus", "human papillomavirus",
    "epstein-barr virus", "helicobacter pylori", "schistosoma",
    "human t-cell lymphotropic virus", "kaposi sarcoma herpesvirus",
    "indoor radon", "formaldehyde in homes", "asbestos in buildings",
    "drinking water contaminants", "food contaminants",
    "processed meats", "red meat", "charred meats", "alcohol",
    "salted fish", "hot beverages", "acrylamide in food",
    "aflatoxins in food", "pesticide residues",
    "tobacco use", "alcohol consumption", "diet", "physical activity",
    "obesity", "reproductive factors", "hormone use",
    "chronic inflammation", "immune suppression", "oxidative stress",
    "dna damage", "genetic mutations", "epigenetic changes",
    "ultraviolet radiation", "sun exposure", "tanning bed use",
    "arsenic in water", "chlorination byproducts", "nitrate in water",
    "radon in water", "industrial contaminants in water"
],
    "hazardous_substances": [
    "aromatic amines",
    "Asbestos", "asbestiform fibres",
    "volatile organic compounds",
    "nitrogen oxides",
    "fine particulates",
    "occupational exposure",
    "radon",
    "passive smoking",
    "sulphuric acid",
    "inorganic arsenic",
    "SO2",
    "benzene",
    "residential radon exposure",
    "cooking oil vapours",
    "chlorine",
    "hypochlorite",
    "chloramine",
    "chlorine dioxine",
    "ozone",
    "Trihalomethanes", "chloroform", "bromodichloromethane", "chlorodibromomethane", "bromoform",
    "Brominated by-products",
    "nitrites & nitrates",
    "radionuclides",
    "formaldehyde",
    "pyrene",
    "fluoranthene",
    "naphthalenes and catechol",
    "Nickel",
    "cadmium",
    "cobalt",
    "Vinyl chloride",
    "Aflatoxin",
    "aromatic amines",
    "ammonia",
    "tar",
    "carbon monoxide",
    "nicotine",
    "trichloromethane",
    "styrene",
    "trichloroethylene",
    "tetrachloroethylene",
    "carbon tetrachloride",
    "chloroform",
    "Organochlorines",
    "creosote",
    "sulfallate",
    "mycotoxins",
    "Ergotism",
    "Beryllium",
    "Chromium",
    "Ethylene oxide",
    "Smoke",
    "Gasoline",
    "Soot",
    "Ionizing radiation",
    "Noise",
    "O3",
    "NO2",
    "volatile organic compounds",
    "benzene",
    "CO",
    "SO2",
    "ultrafine particles",
    "light pollution",
    "desert dust",
    "wildfires",
    "volcano eruptions",
    "air pollution",
    "arsenic",
    "cadmium",
    "lead",
    "mercury",
    "tungsten",
    "antimony",
    "biomass fuels",
    "secondhand smoke",
    "extreme temperature",
    "tobacco",
    "Perfluoroalkyl",
    "Polyfluoro",
    "perfluoro",
    "mercury",
    "Pesticides",
    "lead",
    "polycyclic aromatic hydrocarbons",
    "cadmium",
    "chromium",
    "arsenic",
    "diisocyanates",
    "Tobacco",
    "nitrogen oxide",
    "sulfur dioxide",
    "carbon monoxide",
    "volatile organic compounds",
    "particulate matter",
    "NO2",
    "Coal dust",
    "Livestock farming and agriculture",
    "biological dusts",
    "mineral dusts",
    "gases or fumes",
    "cadmium",
    "barium",
    "cobalt",
    "molybdenum",
    "mercury",
    "cesium",
    "manganese",
    "antimony",
    "lead",
    "tin",
    "strontium",
    "tungsten",
    "thallium",
    "uranium",
    "arsenous acid",
    "arsenic acid",
    "arsenobetaine",
    "arsenocholine",
    "dimethylarsinic acid",
    "monomethylarsonic acid",
    "total arsenic",
    "1-hydroxynaphthalene",
    "2-hydroxynaphthalene",
    "3-hydroxyfluorene",
    "2-hydroxyfluorene",
    "1-hydroxyphenanthrene",
    "1-hydroxypyrene",
    "2 & 3-hydroxyphenanthrene",
    "sulfates",
    "ammonia",
    "sodium chloride",
    "black carbon",
    "mineral dust",
    "Pesticides",
    "dichlorodiphenyltrichloroethane",
    "lindane chlordane",
    "chlorinated hydrocarbons",
    "polycyclic aromatic hydrocarbons",
    "Bisphenol A",
    "Phthalates",
    "butyl benzyl phthalate",
    "Polybrominated Diphenyl Ethers",
    "Arsenic",
    "Cadmium",
    "nitrates",
    "nitrites",
    "N-nitroso",
    "N-nitroso",
    "arsenic",
    "dioxins",
    "talc",
    "straight oil machining fluids",
    "nicotine",
    "dichlorodiphenyldichloroethylene"
]
}


In [ ]:
df.columns

Index(['PMID', 'Title', 'Abstract', 'Authors', 'Affiliation', 'Country',
       'Journal', 'Year', 'DOI', 'URL'],
      dtype='object')

In [ ]:
assert "Abstract" in df.columns, "Column 'abstract' not found!"


In [ ]:
def flatten_dictionary(dictionary):
    terms = set()
    for values in dictionary.values():
        for v in values:
            terms.add(v.lower().strip())
    return terms


In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s\.\!\?]', '', text)
    return text.strip()


In [ ]:
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)


In [ ]:
def filter_sentences(sentences, disease_terms, factor_terms):
    filtered = []

    for sent in sentences:
        sent_lower = sent.lower()

        has_disease = any(term in sent_lower for term in disease_terms)
        has_factor = any(term in sent_lower for term in factor_terms)

        if has_disease and has_factor:
            filtered.append(sent)

    return filtered


In [ ]:

DISEASE_TERMS = flatten_dictionary(FINAL_DISEASE_NAMES)
FACTOR_TERMS = flatten_dictionary(ENVIRONMENTAL_FACTORS)


In [ ]:
tqdm.pandas(desc="Processing abstracts")

cleaned_abstracts = []

for abstract in tqdm(df["Abstract"], desc="Filtering abstracts"):
    cleaned_text = clean_text(abstract)
    sentences = split_sentences(cleaned_text)
    filtered_sentences = filter_sentences(
        sentences,
        DISEASE_TERMS,
        FACTOR_TERMS
    )

    cleaned_abstracts.append(" ".join(filtered_sentences))


Filtering abstracts: 100%|██████████| 27312/27312 [00:41<00:00, 663.96it/s] 


In [ ]:
df["filtered_abstract"] = cleaned_abstracts
df_filtered = df[df["filtered_abstract"].str.strip() != ""]
df_filtered.to_csv("filtered_abstracts_cancer.csv", index=False)



✅ Saved filtered abstracts to filtered_abstracts.csv


In [ ]:
from google.colab import files

files.download("filtered_abstracts_cancer.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install pyahocorasick



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.9 MB/s eta 0:00:00


In [ ]:

!pip install pyahocorasick
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import requests, gzip, xml.etree.ElementTree as ET, re
from tqdm import tqdm
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

tqdm.pandas()

In [ ]:
df = df.dropna(subset=["Abstract"])
df["Abstract"] = df["Abstract"].astype(str)
df = df[df["Abstract"].str.strip() != ""]


In [ ]:

def load_terms_from_file(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return {line.strip().lower() for line in f if len(line.strip()) > 2}
    except:
        return set()


In [ ]:
try:
    url = "https://ftp.ebi.ac.uk/pub/databases/chebi/Flat_file_tab_delimited/names.tsv"
    r = requests.get(url)
    with open("chebi_terms.txt", "w", encoding="utf-8") as f:
        for line in r.text.splitlines()[1:]:
            name = line.split("\t")[1]
            if len(name) > 2:
                f.write(name.lower() + "\n")
except:
    print(" ChEBI download failed, using local file if exists")

In [ ]:
try:
    url = "https://www.ebi.ac.uk/ols/api/ontologies/envo/terms?size=10000"
    data = requests.get(url).json()
    with open("envo_terms.txt", "w", encoding="utf-8") as f:
        for term in data["_embedded"]["terms"]:
            label = term.get("label", "")
            if label:
                f.write(label.lower() + "\n")
except:
    print(" ENVO download failed, using local file if exists")


In [ ]:
try:
    url = "http://ctdbase.org/reports/CTD_chemicals.tsv.gz"
    r = requests.get(url)
    content = gzip.decompress(r.content).decode("utf-8")
    with open("ctd_terms.txt", "w", encoding="utf-8") as f:
        for line in content.splitlines()[1:]:
            name = line.split("\t")[0]
            if len(name) > 2:
                f.write(name.lower() + "\n")
except:
    print("❌ CTD chemicals download failed, using local file if exists")

In [ ]:
chebi_terms = load_terms_from_file("chebi_terms.txt")
envo_terms = load_terms_from_file("envo_terms.txt")
ctd_terms = load_terms_from_file("ctd_terms.txt")
ENV_FACTOR_TERMS = chebi_terms | envo_terms | ctd_terms
print(f" Factor terms loaded: {len(ENV_FACTOR_TERMS)}")


In [ ]:
def load_ctd_diseases():
    url = "http://ctdbase.org/reports/CTD_diseases.tsv.gz"
    r = requests.get(url)
    content = gzip.decompress(r.content).decode("utf-8")
    diseases = set()
    for line in content.splitlines()[1:]:
        diseases.add(line.split("\t")[0].lower())
    return diseases

ctd_diseases = load_ctd_diseases()

In [ ]:
def load_doid_diseases():
    url = "http://purl.obolibrary.org/obo/doid.obo"
    r = requests.get(url)
    diseases = set()
    for line in r.text.splitlines():
        if line.startswith("name:"):
            name = line.split("name:")[1].strip()
            diseases.add(name.lower())
    return diseases

doid_diseases = load_doid_diseases()

In [ ]:
DISEASE_TERMS = ctd_diseases | doid_diseases
print(f"Disease terms loaded: {len(DISEASE_TERMS)}")


In [ ]:
def keep_sentence(sentence):
    sent_lower = sentence.lower()
    for factor in ENV_FACTOR_TERMS:
        if re.search(r"\b" + re.escape(factor) + r"\b", sent_lower):
            return True
    for disease in DISEASE_TERMS:
        if re.search(r"\b" + re.escape(disease) + r"\b", sent_lower):
            return True
    return False

In [ ]:
def reduce_abstract(abstract):
    if not isinstance(abstract, str):
        return ""
    sentences = sent_tokenize(abstract)
    kept = [s for s in sentences if keep_sentence(s)]
    return " ".join(kept)

In [ ]:
df["filtered_abstract"] = df["Abstract"].progress_apply(reduce_abstract)
df = df[df["filtered_abstract"].str.strip() != ""]

print("Remaining rows:", len(df))
print("\nExample filtered abstract:\n", df["filtered_abstract"].iloc[0])

In [ ]:
filtered_list

['Passive smoking in the paediatric age group is associated with an increased frequency of a number of childhood respiratory disorders. However, its effect on ciliary regeneration after functional endoscopic sinus surgery for chronic sinusitis has not previously been reported. We conducted a prospective, nonrandomised cohort study on 38 paediatric patients with chronic sinusitis.',
 'Hepatitis C virus (HCV) is a Flaviviridae virus transmitted through contact with infected bodily fluids or utilization of non-sterile instruments. Exposure to metals is common and difficult to avoid in the environment because of their widespread use in multiple industries. In this study, barium (Ba), cadmium (Cd), cobalt (Co), manganese (Mn), molybdenum (Mo), lead (Pb), antimony (Sb), thallium (Tl), tin (Sn), and tungsten (W) were analyzed with their association on HCV infection prevalence. Tungsten (W) Q3 and Q4 showed a higher prevalence of HCV compared to W values below the detection limit (dl) with ORs

In [ ]:
filtered_list


["The methods are applied to the Nutritional Biomarkers Study of the Women's Health Initiative.",
 'Exposure to metals is common and difficult to avoid in the environment because of their widespread use in multiple industries. In this study, barium (Ba), cadmium (Cd), cobalt (Co), manganese (Mn), molybdenum (Mo), lead (Pb), antimony (Sb), thallium (Tl), tin (Sn), and tungsten (W) were analyzed with their association on HCV infection prevalence. Tungsten (W) Q3 and Q4 showed a higher prevalence of HCV compared to W values below the detection limit (dl) with ORs 13.623 and 11.687, respectively. Molybdenum (Mo) showed significant increases in ORs of being HCV-positive across quartiles Q2, Q3, and Q4 (ORs 7.186, 5.472, and 8.579, respectively), compared to the lowest quartile (Q1) of Mo.',
 'Ambient carbon monoxide (CO) has been linked with mortality and morbidity.',
 'We followed a cohort of 136 beryllium oxide ceramics workers from 1992 to 2003, including those who left employment, for b

In [ ]:
print(df["Abstract"].tolist())

["Biomedical researchers are often interested in estimating the effect of an environmental exposure in relation to a chronic disease endpoint. However, the exposure variable of interest may be measured with errors. In a subset of the whole cohort, a surrogate variable is available for the true unobserved exposure variable. The surrogate variable satisfies an additive measurement error model, but it may not have repeated measurements. The subset in which the surrogate variables are available is called a calibration sample. In addition to the surrogate variables that are available among the subjects in the calibration sample, we consider the situation when there is an instrumental variable available for all study subjects. An instrumental variable is correlated with the unobserved true exposure variable, and hence can be useful in the estimation of the regression coefficients. In this paper, we propose a nonparametric method for Cox regression using the observed data from the whole cohor